# Cyclone tracks generation

This notebook was developped to run multiple scenarios and experiments by generating the fitting config file and running the synth task. A try / except condition can be added to test multiple scenarios / experiments automatically.

## Prerequisites

Before running it, the following conditions should be met:

- Cmip6 climatic data from the scenarios / experiments to run should be loaded in the `data/input/cmip6_data` folder. This can be done with the notebook 'prepare_cmip6_data.ipynb', or directly by running the pangeo task.

- Historical data for the scenario considered should also be downloaded. This can also be done from the notebook 'prepare_cmip6_data.ipynb', by selecting the same scenario and experiment historical.

- The natural earth datasets with coastline and land shapes should be loaded in the `data/input` folder. To do so, go to https://www.quickmaptools.com/download-natural-earth and dowload coastline and land data by selecting the scale 1:10m and Shapefile (ZIP) data.

- The file used for bias correction (named `ERA5_benchmark_100tracks_bias_corrections.csv`), computed on ERA5 data, should be aded in `data/input/bias_correction`.

- Historical cyclone tracks data (from NOAA) should be downloaded from the following link : https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/. The file name is  `ibtracs.since1980.list.v04r01.csv`. It should then placed in `data/input`.

- Fit coefficients should be placed in the folder `input/fit`.

The fit coefficients and ERA5 data for bias correction are located in the folder P:\Projets Internes\PLADIFES common goods\CATHERINA.


### Warning

To run the simulation on an experiment, the model should first be run on the historical data from the same scenario.

In [1]:
import toml
import subprocess
import time

Definition of scenarios and experiments to run

In [2]:
# scenarios = {
#     'ACCESS-CM2':[["historical"], ["ssp245"], ["ssp370"], ["ssp585"]],
#     'FGOALS-g3':[["historical"], ["ssp370"], ["ssp585"]],
#     'IPSL-CM6A-LR':[["historical"], ["ssp370"], ["ssp585"]],
#     'MPI-ESM1-2-LR':[["historical"], ["ssp370"], ["ssp585"]],
# }

scenarios = {
    'ACCESS-CM2':[["historical"]],
}

Main loop

In [3]:
# The total number of seeds run is equel to: number_times_running_seeds * number_seeds_per_run.
# You should first increase number_seeds_per_run to optimize the calculations.

number_times_running_seeds = 1
number_seeds_per_run = 1

time_dict = {}

def get_dates(experiment):
    # Retrieve the start and end years for the given experiment, depending on if it's historical or future experiment
    if experiment == "historical":
        start_year = 1980
        end_year = 2015
    else:
        start_year = 2025
        end_year = 2100
    return start_year, end_year

for model, experiments in scenarios.items():

    time_dict.setdefault(model, {})

    for experiment in experiments:
        print(experiment)

        ## Generate config file ##

        start_year, end_year = get_dates(experiment[0])

        with open("../config_model.toml", "r") as f:
            config = toml.load(f)

        config["main_params"]["models"] = [model]
        config["main_params"]["experiments"] = experiment
        config["main_params"]["start_year"] = start_year
        config["main_params"]["end_year"] = end_year
        config["main_params"]["n_seeds"] = number_seeds_per_run

        with open("../config.toml", "w") as f:
            toml.dump(config, f)
        
        ## Run experimnent ##

        print(f"Running with: {model} {experiment[0]}" )

        time_dict[model][experiment[0]] = []

        for i in range(number_times_running_seeds):
            print(f" - Running seeds: {i}")
            start_time = time.perf_counter()

            result = subprocess.run(
                ["pixi", "run", "synth"],
                capture_output=True,
                text=True
            )

            print("STDOUT:\n", result.stdout)
            print("STDERR:\n", result.stderr)
            print("RETURN CODE:", result.returncode)

            elapsed = time.perf_counter() - start_time
            print(f"--- {elapsed:.2f} seconds ---")
            time_dict[model][experiment[0]].append(elapsed)

['historical']
Running with: ACCESS-CM2 historical
 - Running seeds: 0
STDOUT:
 [06/25/26 18:15:33] INFO     Running Catherina module. | synthetic_tracks.py |  
                             line 39                                            
Starting at seed number:  1
--- Correcting climate bias ---
intensifying tracks...
['SID', 'step', 'lat_left', 'lon_left', 'datetime', 'time', 'y', 'x', 'nshr', 'MSLP', 'T_strat', 'SST', 'height', 'lat_right', 'lon_right', 'distance_track_env', 'thermo_eff', 'seed', 'year', 'month', 'geometry']
['basin', 'geometry']

STDERR:
  WARN Encountered ambiguous version specifier `2024.11.2`, could be `2024.11.2.*` but assuming you meant `==2024.11.2`. In the future this will result in an error.
✨ Pixi task (synth): python synthetic_tracks.py

100%|██████████| 96/96 [01:17<00:00,  1.24it/s]


TC genesis for groups (basin, seed, month):   0%|          | 0/72 [00:00<?, ?it/s]

TC genesis for groups (basin, seed, month):  14%|=         | 10/72 [00:00<00:00, 92